# Ensemble Learning

## 1. Gradient Boosting Tree

### 1.1. GBT Regressor

In [1]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

# Load regression dataset https://archive.ics.uci.edu/dataset/186/wine+quality
dataset = fetch_ucirepo(id=186)
X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel() # flatten to 1D array

In [2]:
# TODO: Split into train and test sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train shape:", X_train_reg.shape)
print("X_test shape:", X_test_reg.shape)

X_train shape: (5197, 11)
X_test shape: (1300, 11)


In [3]:
# TODO: Create the GBT Regressor class
class MyGBTRegressor:
    def __init__(self,
                 n_estimators=100,
                 learning_rate=0.1,
                 max_depth=3,
                 random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state
        self.initial_pred = None
        self.trees = []

    def initialize_prediction(self, y):
        return np.full(y.shape[0], np.mean(y))

    def compute_negative_gradient(self, y_true, y_pred):
        """Negative gradient of MSE loss (residuals)."""
        return y_true - y_pred

    def fit(self, X, y):
        self.initial_pred = self.initialize_prediction(y)
        y_pred = self.initial_pred.copy()
        rng = np.random.RandomState(self.random_state)
        self.trees = []

        for _ in range(self.n_estimators):
            residuals = self.compute_negative_gradient(y, y_pred)
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=rng.randint(0, 10_000)
            )
            tree.fit(X, residuals)

            update = tree.predict(X)
            y_pred += self.learning_rate * update
            self.trees.append(tree)

        return self

    def predict(self, X):
        y_pred = np.full(X.shape[0], np.mean(self.initial_pred))
        for tree in self.trees:
            update = tree.predict(X)
            y_pred += self.learning_rate * update
        return y_pred

In [4]:
# TODO: Train the GBT Regressor and evaluate its performance
my_regressor = MyGBTRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
my_regressor.fit(X_train_reg, y_train_reg)
y_pred_reg = my_regressor.predict(X_test_reg)

my_reg_mse = mean_squared_error(y_test_reg, y_pred_reg)
print(f"MyGBTRegressor MSE: {my_reg_mse:.4f}")

MyGBTRegressor MSE: 0.4798


In [5]:
# TODO: Compare with sklearn's GradientBoostingRegressor
sk_regressor = GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
sk_regressor.fit(X_train_reg, y_train_reg)
sk_y_pred_reg = sk_regressor.predict(X_test_reg)

sk_reg_mse = mean_squared_error(y_test_reg, sk_y_pred_reg)
print(f"Sklearn GradientBoostingRegressor MSE: {sk_reg_mse:.4f}")
print(f"Difference (MyGBT - Sklearn): {my_reg_mse - sk_reg_mse:+.4f}")

Sklearn GradientBoostingRegressor MSE: 0.4794
Difference (MyGBT - Sklearn): +0.0004


### 1.2. GBT Classifier

In [6]:
# Load classification dataset https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
dataset = fetch_ucirepo(id=17)
X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel() # flatten to 1D array
y = (y == 'M').astype(int)  # Convert labels to 0 and 1

In [7]:
# TODO: Split into train and test sets
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X_train shape:", X_train_cls.shape)
print("X_test shape:", X_test_cls.shape)

X_train shape: (455, 30)
X_test shape: (114, 30)


In [8]:
# TODO: Create the GBT Classifier class
class MyGBTClassifier:
    def __init__(self,
                 n_estimators=100,
                 learning_rate=0.1,
                 max_depth=3,
                 random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state
        self.initial_log_odds = None
        self.trees = []

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def initialize_prediction(self, y):
        # log-odds of the positive class
        p_mean = np.mean(y)
        return np.log(p_mean / (1.0 - p_mean))

    def compute_negative_gradient(self, y_true, proba_pred):
        return y_true - proba_pred

    def fit(self, X, y):
        y = np.asarray(y).astype(float)
        self.initial_log_odds = self.initialize_prediction(y)
        f_pred = np.full(y.shape[0], self.initial_log_odds)
        rng = np.random.RandomState(self.random_state)
        self.trees = []

        for m in range(self.n_estimators):
            p_pred = self._sigmoid(f_pred)
            residuals = self.compute_negative_gradient(y, p_pred)

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=rng.randint(0, 10_000)
            )
            tree.fit(X, residuals)

            # Update log-odds: f_{m+1} = f_m + eta * h_m(x)
            update = tree.predict(X)
            f_pred += self.learning_rate * update
            self.trees.append(tree)

        return self

    def _raw_score(self, X):
        f_pred = np.full(X.shape[0], self.initial_log_odds)
        for tree in self.trees:
            f_pred += self.learning_rate * tree.predict(X)
        return f_pred

    def predict_proba(self, X):
        f_pred = self._raw_score(X)
        p1 = self._sigmoid(f_pred)
        p0 = 1.0 - p1
        return np.vstack([p0, p1]).T

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] >= 0.5).astype(int)

In [9]:
# TODO: Train the GBT Classifier and evaluate its performance
my_clf = MyGBTClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
my_clf.fit(X_train_cls, y_train_cls)

y_pred_cls = my_clf.predict(X_test_cls)
y_pred_proba_cls = my_clf.predict_proba(X_test_cls)[:, 1]

my_accuracy = accuracy_score(y_test_cls, y_pred_cls)
my_f1 = f1_score(y_test_cls, y_pred_cls)
my_roc_auc = roc_auc_score(y_test_cls, y_pred_proba_cls)

print(f"MyGBTClassifier Accuracy:  {my_accuracy:.4f}")
print(f"MyGBTClassifier F1 Score:  {my_f1:.4f}")
print(f"MyGBTClassifier ROC AUC:   {my_roc_auc:.4f}")

MyGBTClassifier Accuracy:  0.9474
MyGBTClassifier F1 Score:  0.9231
MyGBTClassifier ROC AUC:   0.9821


In [10]:
# TODO: Compare with sklearn's GradientBoostingClassifier
sk_clf = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42)
sk_clf.fit(X_train_cls, y_train_cls)

sk_y_pred_cls = sk_clf.predict(X_test_cls)
sk_y_pred_proba_cls = sk_clf.predict_proba(X_test_cls)[:, 1]

sk_accuracy = accuracy_score(y_test_cls, sk_y_pred_cls)
sk_f1 = f1_score(y_test_cls, sk_y_pred_cls)
sk_roc_auc = roc_auc_score(y_test_cls, sk_y_pred_proba_cls)

print(f"Sklearn GradientBoostingClassifier Accuracy:  {sk_accuracy:.4f}")
print(f"Sklearn GradientBoostingClassifier F1 Score:  {sk_f1:.4f}")
print(f"Sklearn GradientBoostingClassifier ROC AUC:   {sk_roc_auc:.4f}")

Sklearn GradientBoostingClassifier Accuracy:  0.9561
Sklearn GradientBoostingClassifier F1 Score:  0.9383
Sklearn GradientBoostingClassifier ROC AUC:   0.9964


## 2. XGBoost

In [ ]:
# TODO: Create XGBoost Regressor and Classifier models and compare their performance with your implementations
import xgboost as xgb

# --- 1. XGBoost Regressor (Wine Quality) ---
xgb_reg = xgb.XGBRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
xgb_reg.fit(X_train_reg, y_train_reg)
xgb_y_pred_reg = xgb_reg.predict(X_test_reg)
xgb_reg_mse = mean_squared_error(y_test_reg, xgb_y_pred_reg)

print("=== Regression Results (Wine Quality) ===")
print(f"MyGBTRegressor MSE:          {my_reg_mse:.4f}")
print(f"Sklearn GBT Regressor MSE:   {sk_reg_mse:.4f}")
print(f"XGBoost Regressor MSE:       {xgb_reg_mse:.4f}")

# --- 2. XGBoost Classifier (Breast Cancer) ---
xgb_clf = xgb.XGBClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss'
)
xgb_clf.fit(X_train_cls, y_train_cls)
xgb_y_pred_cls = xgb_clf.predict(X_test_cls)
xgb_y_pred_proba_cls = xgb_clf.predict_proba(X_test_cls)[:, 1]

xgb_accuracy = accuracy_score(y_test_cls, xgb_y_pred_cls)
xgb_f1 = f1_score(y_test_cls, xgb_y_pred_cls)
xgb_roc_auc = roc_auc_score(y_test_cls, xgb_y_pred_proba_cls)

print("\n=== Classification Results (Breast Cancer) ===")
print(f"MyGBTClassifier     -> Accuracy: {my_accuracy:.4f}, F1: {my_f1:.4f}, ROC AUC: {my_roc_auc:.4f}")
print(f"Sklearn GBT         -> Accuracy: {sk_accuracy:.4f}, F1: {sk_f1:.4f}, ROC AUC: {sk_roc_auc:.4f}")
print(f"XGBoost Classifier  -> Accuracy: {xgb_accuracy:.4f}, F1: {xgb_f1:.4f}, ROC AUC: {xgb_roc_auc:.4f}")

=== Regression Results (Wine Quality) ===
MyGBTRegressor MSE:          0.4798
Sklearn GBT Regressor MSE:   0.4794
XGBoost Regressor MSE:       0.4789

=== Classification Results (Breast Cancer) ===
MyGBTClassifier     -> Accuracy: 0.9474, F1: 0.9231, ROC AUC: 0.9821
Sklearn GBT         -> Accuracy: 0.9561, F1: 0.9383, ROC AUC: 0.9964
XGBoost Classifier  -> Accuracy: 0.9561, F1: 0.9367, ROC AUC: 0.9960


: 